# Clustering: nulls, stability, and independent validation

**Accompanies Section 8 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

Every clustering algorithm returns clusters, because the algorithm has no way to decline. This notebook shows how to establish that the clusters you obtained correspond to something, using a null model, a stability analysis, an independent order parameter that was not used to obtain them, and a comparison against the other choices you could defensibly have made.

### Learning objectives
- Test whether data is clusterable *before* clustering it
- See why maximizing an index over k is a selection procedure
- Measure per-cluster bootstrap stability
- Validate clusters of an MD trajectory against an independent physical observable
- Separate the two questions "**are** there clusters?" and "**what** are the clusters?"
- Measure agreement between partitions with the adjusted Rand index and a permutation test
- Learn the fingerprint-specific pitfalls: bit vs count, MACCS, metric choice, size dependence

### What this notebook is designed to make go wrong
An apparently optimal number of clusters found in data that has no clusters, and a silhouette scan that gives no hint anything is wrong. Then, in the case study, a data set that passes a clusterability test under three representations and yields three partitions that agree with each other at close to chance.

### What you need installed
Sections 1 to 5 need numpy and scikit-learn. Section 6 also needs ASE, which reads the QM7 structure file, and it reproduces manuscript Figure 8 exactly. Section 7 uses RDKit.

### Roughly how long it takes
Ten minutes or so. The bootstrap stability in section 3 and the three representations in section 6 are the slow parts.

In [ ]:
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

## 1. Should you cluster this at all?

Clusterability is the question that should precede clustering, and it almost never does. We
compare a structureless data set against one with genuine clusters, using a permutation null:
shuffling each feature independently destroys joint structure while preserving every marginal
distribution exactly.

### Build the null by shuffling each feature independently

`permutation_null` shuffles each column of the data independently. That destroys
the joint structure, which is what a cluster is, and leaves every single
feature's distribution exactly as it was. The null data set therefore has your
marginals and no clusters, so anything you find above it is joint structure and
not a skewed histogram seen from the side.

`clusterability` runs k-means on the real data, records the mean silhouette,
then does the same on each permuted copy. You get one observed number and a
null distribution to read it against. The p-value uses the (r + 1) / (n + 1)
form, so with 99 replicates the smallest value it can report is 0.01, which is
honest about the resolution of the test.

`describe_null` and `plot_null_comparison` do the reporting: one line of
text, and a histogram of the null with the observed value drawn on it.

The effect that line reports in SD is a z-score: how many standard
deviations the observed value sits above the mean of the null, printed later
in the notebook as "SD above null". Below about 2 is unconvincing, and a very
large value means the null was easy to beat and not that the clusters mean
something. Beating the null establishes that the features are not independent
of one another, which is much weaker than establishing that groups exist.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def permutation_null(X, rng):
    """Shuffle each feature independently, destroying joint structure only."""
    Xp = np.array(X, copy=True, dtype=float)
    for j in range(Xp.shape[1]):
        rng.shuffle(Xp[:, j])
    return Xp


def clusterability(X, n_clusters=2, n_replicates=199, random_state=SEED):
    """Silhouette of a k-means fit on X, and the same statistic on permuted copies.

    Returns (observed, null_values). Run it before you choose k, not after.
    """
    gen = np.random.default_rng(random_state)

    def statistic(data):
        labels = KMeans(n_clusters=n_clusters, n_init=10,
                        random_state=random_state).fit_predict(data)
        return float(silhouette_score(data, labels))

    observed = statistic(X)
    null = np.array([statistic(permutation_null(X, gen)) for _ in range(n_replicates)])
    return observed, null


def null_p_value(observed, null):
    """One-sided empirical p-value. The +1 keeps it off an impossible zero."""
    return (int(np.sum(null >= observed)) + 1) / (len(null) + 1)


def describe_null(observed, null):
    """One line: the observed statistic, the null it is read against, the verdict."""
    lo, hi = np.percentile(null, [2.5, 97.5])
    p = null_p_value(observed, null)
    effect = (observed - null.mean()) / null.std(ddof=1)
    verdict = "EXCEEDS null" if p < 0.05 else "within null"
    return (f"observed = {observed:.4f}; null = {null.mean():.4f} [{lo:.4f}, {hi:.4f}]; "
            f"p = {p:.4f}; effect = {effect:+.2f} SD -> {verdict}")


def plot_null_comparison(observed, null, ax, title):
    """The null distribution as a histogram, with the observed value drawn on it."""
    ax.hist(null, bins=25, color=LAVENDER, edgecolor="white", linewidth=0.5,
            label="null distribution")
    exceeds = null_p_value(observed, null) < 0.05
    # Solid against dashed carries the verdict, so the panel reads in grayscale.
    ax.axvline(observed, color=GREEN if exceeds else PLUM, lw=2,
               linestyle="-" if exceeds else (0, (4, 1.5)),
               label="observed (exceeds null)" if exceeds else "observed (within null)")
    lo, hi = np.percentile(null, [2.5, 97.5])
    ax.axvspan(lo, hi, facecolor="none", edgecolor=SLATE, hatch="///", lw=0.4, alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel("statistic")
    ax.set_ylabel("null replicates")
    ax.legend(frameon=False)

In [ ]:
noise = rng.standard_normal((400, 20))

centers = np.array([[0, 0], [8, 1], [3, 7], [-5, 5]], dtype=float)
structured = np.vstack([
    np.hstack([rng.standard_normal((100, 2)) + c, rng.standard_normal((100, 18))])
    for c in centers
])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, data, name in [(axes[0], noise, "structureless"), (axes[1], structured, "four real clusters")]:
    observed, null = clusterability(data, n_clusters=4, n_replicates=199, random_state=SEED)
    plot_null_comparison(observed, null, ax=ax,
                         title=f"{name}: p = {null_p_value(observed, null):.3f}")
    print(f"{name:22s} {describe_null(observed, null)}")
plt.tight_layout()
plt.show()

## 2. Scanning k is a selection procedure

On structureless data the silhouette still has a maximum over k, and nothing in the scan itself
signals that the answer is meaningless. Against a permutation null band the situation is
clear.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = np.arange(2, 13)

def silhouette_curve(data, seed=SEED):
    return np.array([
        silhouette_score(data, KMeans(int(k), n_init=10, random_state=seed).fit_predict(data))
        for k in ks
    ])

def null_band(data, n=199):
    local_rng = np.random.default_rng(SEED)
    curves = np.array([silhouette_curve(permutation_null(data, local_rng), SEED + i) for i in range(n)])
    return np.percentile(curves, [2.5, 97.5], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, data, name in [(axes[0], noise, "structureless"), (axes[1], structured, "four real clusters")]:
    observed = silhouette_curve(data)
    lo, hi = null_band(data)
    ax.fill_between(ks, lo, hi, color="#BFBFBF", alpha=0.7, lw=0, label="95% permutation null (199 shuffles)")
    ax.plot(ks, observed, "o-", ms=4, label="observed")
    ax.set_xlabel("number of clusters k")
    ax.set_title(name)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("mean silhouette")
plt.show()

## 3. Stability: would these clusters reappear in a fresh sample?

An internal index measures compactness under the assumptions it encodes. Bootstrap stability
measures reproducibility, which is much closer to what "are these clusters real?" means.

### Measure stability by bootstrapping and matching clusters

`cluster_stability` is written out here because the details are what decide
whether the number means anything.

Cluster the full data set once; that partition is the reference. Then draw a
bootstrap sample of the same size with replacement, cluster it, and ask how well
each reference cluster survived. Survival is the Jaccard similarity between two
sets of sample indices: the size of the overlap over the size of the union.

Two details in it do real work. The first is that the bootstrap replicate is
clustered with its repeated draws included, since those repeats carry the
resampling weight that makes this a bootstrap and not a 63% subsample, and that
scoring then uses only the samples that were actually drawn, since a sample
never drawn cannot be recovered. The second is that the reference clusters are
matched to the bootstrap clusters by optimal one-to-one assignment and not by
greedy best match, which would let two reference clusters claim the same
bootstrap cluster and push the score up.

Read the result against the usual bands: above 0.85 highly stable, 0.75 to 0.85
stable, 0.60 to 0.75 weak, below 0.60 dissolved. In practice, above about 0.75
you can call a cluster reproducible, 0.60 to 0.75 you report it with its
instability stated, and below 0.60 the cluster is an artifact of which molecules
happened to land in your sample. Read the bands per cluster, since an average
hides the one unstable cluster your story rests on. Those bands are conventions
and not tests, and stability measures reproducibility and not correctness, so a
stable cluster can still be an artifact of a systematic curation error.

In [ ]:
from scipy.optimize import linear_sum_assignment


def jaccard(a, b):
    """Overlap over union, for two sets of sample indices."""
    a, b = np.unique(a), np.unique(b)
    intersection = np.intersect1d(a, b, assume_unique=True).size
    union = a.size + b.size - intersection
    return float(intersection / union) if union else 0.0


def cluster_stability(X, cluster_fn, n_bootstrap=100, random_state=SEED):
    """Per-cluster bootstrap Jaccard stability.

    ``cluster_fn`` maps a data matrix to integer labels and has to accept data
    sets of varying size. Fixing its seed pins the variation you are measuring
    to the resampling instead of the initialization.

    Returns (scores, sizes): scores[b, c] is the Jaccard that original cluster c
    achieved in bootstrap replicate b, and sizes[c] is how many members it has.
    """
    X = np.asarray(X, dtype=float)
    n_samples = X.shape[0]
    gen = np.random.default_rng(random_state)

    reference = np.asarray(cluster_fn(X))
    reference_sets = [np.where(reference == lab)[0] for lab in np.unique(reference)]
    sizes = np.array([s.size for s in reference_sets])

    scores = np.zeros((n_bootstrap, len(reference_sets)))
    for b in range(n_bootstrap):
        idx = gen.choice(n_samples, size=n_samples, replace=True)
        present = np.unique(idx)
        # Cluster the resampled multiset, repeats and all, then map the labels
        # back onto original sample indices to score them.
        boot = np.asarray(cluster_fn(X[idx]))
        boot_sets = [np.unique(idx[boot == lab]) for lab in np.unique(boot)]

        overlap = np.zeros((len(reference_sets), len(boot_sets)))
        for i, ref in enumerate(reference_sets):
            drawn = np.intersect1d(ref, present, assume_unique=True)
            for j, bs in enumerate(boot_sets):
                overlap[i, j] = jaccard(drawn, bs)
        # Hungarian assignment on the overlap matrix: one reference cluster to
        # at most one bootstrap cluster.
        rows, cols = linear_sum_assignment(-overlap)
        scores[b, rows] = overlap[rows, cols]
    return scores, sizes


def overall_stability(scores, sizes):
    """Size-weighted mean, so one tiny unstable cluster cannot run the headline."""
    return float(np.sum(scores.mean(axis=0) * sizes / sizes.sum()))


def print_stability(scores, sizes):
    """Per-cluster verdicts, in the bands quoted above."""
    print(f"overall (size-weighted) stability: {overall_stability(scores, sizes):.3f}")
    for c, (score, size) in enumerate(zip(scores.mean(axis=0), sizes)):
        if score > 0.85:
            verdict = "highly stable"
        elif score >= 0.75:
            verdict = "stable"
        elif score >= 0.60:
            verdict = "weak; report the instability"
        else:
            verdict = "dissolved; do not interpret as a group"
        print(f"  cluster {c} (n={size}): Jaccard {score:.3f}, {verdict}")


def plot_stability(scores, sizes, ax, threshold=0.75):
    """One box per original cluster: the spread of its Jaccard over replicates."""
    per_cluster = scores.mean(axis=0)
    boxes = ax.boxplot(
        [scores[:, c] for c in range(scores.shape[1])],
        positions=np.arange(scores.shape[1]),
        widths=0.6,
        patch_artist=True,
        medianprops=dict(color=SLATE, lw=1.6),
        flierprops=dict(marker="o", ms=2.5, markerfacecolor=SLATE,
                        markeredgecolor="none", alpha=0.4),
    )
    for c, patch in enumerate(boxes["boxes"]):
        stable = per_cluster[c] >= threshold
        patch.set_facecolor(GREEN if stable else PLUM)
        # Hatching, not color, is what survives a black-and-white print.
        patch.set_hatch("" if stable else "xxx")
        patch.set_edgecolor(SLATE)
        patch.set_linewidth(0.9)
        patch.set_alpha(0.55)

    ax.axhline(threshold, color=SLATE, ls="--", lw=1)
    ax.set_xticks(np.arange(scores.shape[1]))
    ax.set_xticklabels([f"{c}\n(n={s})" for c, s in enumerate(sizes)])
    ax.set_xlabel("cluster")
    ax.set_ylabel("bootstrap Jaccard stability")
    ax.set_ylim(0, 1.02)

In [ ]:
def kmeans4(data):
    return KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(data)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, data, name in [(axes[0], structured, "four real clusters"), (axes[1], noise, "structureless")]:
    scores, sizes = cluster_stability(data, kmeans4, n_bootstrap=60, random_state=SEED)
    plot_stability(scores, sizes, ax=ax)
    ax.set_title(f"{name}\n(overall {overall_stability(scores, sizes):.2f})")
    print(f"--- {name} ---")
    print_stability(scores, sizes)
    print()
plt.tight_layout()
plt.show()

## 4. Cluster an aspirin trajectory and validate it against an observable

Here we take a real case: we cluster an MD trajectory and then validate the clusters against
an independent, interpretable physical observable (the dihedral angle between the benzene ring
and the carboxyl group), which was **not** used to obtain them.

That last clause is what makes this validation and not confirmation.

In [ ]:
# The MD17 aspirin trajectory: R positions in angstrom, z atomic numbers, E energies. The
# shipped 10,000 frames are evenly spaced across the full trajectory, so fast observables are
# decorrelated (the energy has statistical inefficiency around 2, measured below), but the slow
# ring-carboxyl dihedral is not: it interconverts slowly, so successive shipped frames stay
# strongly correlated in exactly the coordinate the clustering cares about. Striding by 10 cuts
# the compute tenfold while barely changing the roughly 30 effectively-independent conformational
# samples, which is what the population split really rests on.
raw = np.load(DATA / "md17_aspirin_10000.npz")
STRIDE = 10
positions = raw["R"][::STRIDE]
energies = raw["E"].ravel()[::STRIDE]
n_frames, n_at, _ = positions.shape
print(f"MD17 aspirin: {n_frames} frames of {n_at} atoms (every {STRIDE}th of {len(raw['R'])})")

# Pairwise distances: invariant to rotation and translation by construction,
# which avoids having to choose an alignment procedure at all.
iu = np.triu_indices(n_at, k=1)
features = np.array([
    np.linalg.norm(frame[iu[0]] - frame[iu[1]], axis=1) for frame in positions
])
print(f"feature matrix: {features.shape}")

In [ ]:
observed, null = clusterability(features, n_clusters=2, n_replicates=199, random_state=SEED)
print("Is the trajectory clusterable?")
print("  " + describe_null(observed, null))

sil = np.array([
    silhouette_score(features, KMeans(int(k), n_init=10, random_state=SEED).fit_predict(features))
    for k in range(2, 8)
])
for k, s in zip(range(2, 8), sil):
    print(f"  k = {k}: silhouette {s:.3f}")

# Section 2 warned that taking the argmax of a silhouette scan is a selection
# procedure. It still is. Two things make it defensible here and neither is the
# silhouette: the clusterability test above already established that there is
# structure to find, and the choice of k gets checked in section 4 against a
# dihedral angle the clustering never saw. The full scan is printed above so you
# can see how sharp the maximum is, which is the part a single number hides.
best_k = int(np.arange(2, 8)[np.argmax(sil)])
labels = KMeans(best_k, n_init=10, random_state=SEED).fit_predict(features)
print(f"\nproceeding with k = {best_k}")

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

frac = np.bincount(labels)[np.argmax(np.bincount(labels))] / len(labels)
print(f"larger cluster holds               {frac:8.1%}   article: ~70%")
assert best_k == 2, f"aspirin now splits into {best_k} clusters, not 2"
assert 0.62 <= frac <= 0.78, f"aspirin split drifted to {frac:.1%}/{1 - frac:.1%}"
print("within tolerance")

In [ ]:
scores, sizes = cluster_stability(
    features,
    lambda d: KMeans(best_k, n_init=10, random_state=SEED).fit_predict(d),
    n_bootstrap=40,
    random_state=SEED,
)
print_stability(scores, sizes)

### Check the clusters against a dihedral they never saw

Aspirin's carboxyl group rotates relative to the benzene ring. If the clusters correspond to
distinct conformational basins, the dihedral distributions should separate, and we never gave
the clustering any information about this angle.

In [ ]:
# Atom indices in the MD17 aspirin ordering, read off the connectivity of the
# first frame: the benzene ring, the carboxyl carbon and its carbonyl oxygen.
# The dihedral C6-C5-C10-O7 is the rotation of the carboxyl group out of the
# plane of the ring: the order parameter the clustering never saw. These are
# the same four atoms the manuscript figure uses; picking "the first four
# carbons" instead gives a ring torsion, which is rigid and separates nothing.
ASPIRIN_DIHEDRAL = (6, 5, 10, 7)


def dihedral(p0, p1, p2, p3):
    """Signed dihedral angle in degrees, vectorized over frames."""
    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2
    b1n = b1 / np.linalg.norm(b1, axis=-1, keepdims=True)
    v = b0 - (b0 * b1n).sum(-1, keepdims=True) * b1n
    w = b2 - (b2 * b1n).sum(-1, keepdims=True) * b1n
    return np.degrees(np.arctan2((np.cross(b1n, v) * w).sum(-1), (v * w).sum(-1)))


i, j, k, m = ASPIRIN_DIHEDRAL
angles = dihedral(positions[:, i], positions[:, j], positions[:, k], positions[:, m])

fig, ax = plt.subplots(figsize=(6, 3.2))
for c in range(best_k):
    ax.hist(angles[labels == c], bins=45, range=(-180, 180), alpha=0.6, density=True,
            label=f"cluster {c} (n={np.sum(labels == c)})")
ax.set_xlabel("dihedral angle / degrees")
ax.set_ylabel("probability density")
ax.set_title("Clusters separate along a physical coordinate they never saw")
ax.legend(frameon=False)
plt.show()

In [ ]:
# Item 8c: is "decorrelated by construction" true? Measure it on ALL shipped frames.
def _lag1(x):
    x = np.asarray(x, dtype=float) - np.mean(x)
    return float((x[:-1] * x[1:]).sum() / (x * x).sum())


def _statistical_inefficiency(x, maxlag=500):
    x = np.asarray(x, dtype=float) - np.mean(x)
    n, v, g = len(x), (x * x).mean(), 1.0
    for t in range(1, maxlag):
        c = (x[:-t] * x[t:]).mean() / v
        if c <= 0:
            break
        g += 2 * c * (1 - t / n)
    return float(g)


_full = raw["R"]
_full_cos = np.cos(np.deg2rad(dihedral(_full[:, i], _full[:, j], _full[:, k], _full[:, m])))
_full_E = raw["E"].ravel()
print("Autocorrelation of the shipped frames, before any striding:")
print(f"  energy   : lag-1 {_lag1(_full_E):+.3f}, statistical inefficiency g = {_statistical_inefficiency(_full_E):.0f}")
print(f"  dihedral : lag-1 {_lag1(_full_cos):+.3f}, statistical inefficiency g = {_statistical_inefficiency(_full_cos):.0f}")
print(f"So the energy is decorrelated (g about 2) but the dihedral is not: only about")
print(f"{len(_full_E) / _statistical_inefficiency(_full_cos):.0f} of the 10,000 frames are independent in that coordinate,")
print("which is why the stride costs little and why the population split below is approximate.")

In [ ]:
# Both clusters are planar; they differ in whether the carboxyl points syn (near 0 deg,
# eclipsing the ring) or anti (near 180 deg), so "near-planar vs not" is the wrong label.
# A dihedral is a circular variable, so summarize it with the circular mean, whose samples
# near +/-180 would otherwise average to nonsense.
def circular_mean(deg):
    r = np.deg2rad(deg)
    return float(np.degrees(np.arctan2(np.sin(r).mean(), np.cos(r).mean())))


def circular_separation(a, b):
    gap = abs(circular_mean(a) - circular_mean(b)) % 360
    return min(gap, 360 - gap)


def _wrap180(a):
    return (a + 180.0) % 360.0 - 180.0


centers = {c: circular_mean(angles[labels == c]) for c in range(best_k)}
pops = {c: float(np.mean(labels == c)) for c in range(best_k)}
syn = min(range(best_k), key=lambda c: abs(_wrap180(centers[c] - 0.0)))
anti = min(range(best_k), key=lambda c: abs(_wrap180(centers[c] - 180.0)))
sep = circular_separation(angles[labels == 0], angles[labels == 1])

print(f"syn  cluster (carboxyl near   0 deg): centre {centers[syn]:+7.1f} deg, {100 * pops[syn]:5.1f}% of frames")
print(f"anti cluster (carboxyl near 180 deg): centre {centers[anti]:+7.1f} deg, {100 * pops[anti]:5.1f}% of frames")
print(f"separation {sep:.1f} deg (both are planar; they differ by syn vs anti, not by planarity)")

# Temperature and per-cluster energies, so the split can be sanity-checked against a Boltzmann
# estimate. MD17 aspirin was generated at 500 K (Chmiela et al. 2017); the .npz stores no
# thermostat temperature, so this is the documented value rather than one derived here.
T_KELVIN = 500
for tag, c in (("syn", syn), ("anti", anti)):
    e = energies[labels == c]
    print(f"  {tag:4s} mean energy {e.mean():+.1f} +/- {e.std():.1f} kcal/mol (n={e.size})")
print(f"trajectory temperature: {T_KELVIN} K (documented for MD17 aspirin)")
print("The two energy distributions overlap heavily, and with about thirty effectively-")
print("independent conformational samples the roughly 70/30 split is not pinned down tightly")
print("enough to read against a Boltzmann factor; report it as approximate.")

assert sep > 150, (
    f"the independent order parameter no longer separates the clusters ({sep:.1f} deg); "
    "Figure 7 depends on this"
)
larger = max(pops.values())
assert 0.60 <= larger <= 0.80, f"cluster balance drifted: {larger:.3f}"
print("within tolerance")

## 5. Match the algorithm to what it assumes about your clusters

The article's clustering table gives an "assumes" column and a "how it fails"
column. This cell makes the second one observable. The data below contains three
groups that are unambiguously real, but each breaks a different assumption:
one is elongated instead of spherical, the three differ in size by a factor of
five, and they differ in density.

**Watch the ARI against ground truth.** The adjusted Rand index measures how far
two partitions of the same objects agree, corrected for the agreement chance
alone produces: 1 is identical, 0 is chance, and a negative value is worse than
chance. Every method returns three clusters and every method looks like it
worked, but only one of them recovered the groups.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

rng_alg = np.random.default_rng(SEED)
elongated = rng_alg.normal(0, 1, size=(200, 2)) @ np.array([[6.0, 0.0], [0.0, 0.30]])
small = rng_alg.normal(0, 0.35, size=(40, 2)) + np.array([2.0, 4.0])
diffuse = rng_alg.normal(0, 1.30, size=(60, 2)) + np.array([-6.0, -4.0])
Xa = np.vstack([elongated, small, diffuse])
truth = np.repeat([0, 1, 2], [200, 40, 60])

families = {
    "k-means": KMeans(3, n_init=10, random_state=SEED).fit_predict(Xa),
    "hierarchical (ward)": AgglomerativeClustering(3).fit_predict(Xa),
    "hierarchical (average)": AgglomerativeClustering(3, linkage="average").fit_predict(Xa),
    "Gaussian mixture": GaussianMixture(3, covariance_type="full",
                                        random_state=SEED).fit_predict(Xa),
    "DBSCAN (eps=0.8)": DBSCAN(eps=0.8, min_samples=5).fit_predict(Xa),
}

print(f"{'family':<24s} {'k found':>8s} {'noise':>6s} {'ARI vs truth':>13s}")
for name, lab in families.items():
    core = lab != -1
    print(f"{name:<24s} {len(set(lab[core].tolist())):8d} {int((~core).sum()):6d} "
          f"{adjusted_rand_score(truth, lab):13.2f}")

names = list(families)
print("\npairwise ARI between the families (no ground truth needed):")
print(f"{'':<24s}" + "".join(f"{n.split()[0][:8]:>10s}" for n in names))
for a in names:
    print(f"{a:<24s}" + "".join(
        f"{adjusted_rand_score(families[a], families[b]):10.2f}" for b in names))

print("\nk-means and Ward impose spherical, similarly sized groups, so they cut the")
print("elongated cluster in half and glue the halves to their neighbors. The")
print("mixture model, which allows elliptical covariance, recovers the truth.")
print("The pairwise table alone, which is all you have on real data, already")
print("tells you the answer is contingent on the algorithm.")

---
## 6. Case study: cluster the same molecules under three representations

We use the QM7 molecules with exactly 15 atoms (avoiding the zero-padding artifact discussed in
Section 6 of the article) and build three representations, each of which a chemist could defend.

### Read QM7, and mind the units and the atom count

`qm7.xyz` is an extended-XYZ file, so ASE reads it in one call: `read(path,
index=":")` gives a list of `Atoms` objects, one per molecule, each carrying
positions, atomic numbers and whatever the comment line recorded in
`atoms.info`.

There are two things to know about this particular file. The first is units:
the positions are in **Bohr**, not angstrom, because the conversion from the
original `qm7.mat` carried the atomic units over unchanged. Nothing below is
affected, since every representation here either counts atoms or uses
off-diagonal Coulomb entries, and rescaling all distances by one factor
multiplies those by one constant.
Multiply by 0.529177210903 before doing anything with an absolute length scale,
such as setting a SOAP or ACSF cutoff.

The second is size: molecules differ in atom count, so a fixed-width coordinate
array has to be zero-padded, and those zeros then sit inside every distance you
compute. Taking the molecules with exactly 15 atoms, and only those, avoids the
padding altogether (Section 6 of the article).

In [ ]:
from ase.io import read

frames = read(str(DATA / "qm7.xyz"), index=":")
fifteen = [atoms for atoms in frames if len(atoms) == 15]
positions = np.array([atoms.get_positions() for atoms in fifteen])
numbers = np.array([atoms.get_atomic_numbers() for atoms in fifteen])
print(f"{len(frames)} molecules in the file, {len(fifteen)} of them with exactly 15 atoms")
n_orderings = len({tuple(int(z) for z in num) for num in numbers})
print(f"{n_orderings} distinct atom orderings among them; the Coulomb descriptor below is built")
print("in each file's given atom order, with no canonical sort (see the note that follows).")

The Coulomb matrix is not permutation invariant, so building it from the file's given atom order,
as the cell above reports it does across many distinct orderings, is exactly the practice the
guide's Section 6.1 checklist warns against. That is deliberate here: this case study is the
cautionary example, and Section 6.3 below adds a robustness check that re-runs the algorithm
comparison on a canonically sorted matrix to show whether the conclusion depends on the ordering.

In [ ]:
ANGSTROM_PER_BOHR = 0.529177210903


def coulomb_offdiag(pos, num):
    z = num.astype(float)
    d = np.linalg.norm(pos[:, None] - pos[None, :], axis=-1)
    np.fill_diagonal(d, np.inf)
    iu = np.triu_indices(len(z), 1)
    return (np.outer(z, z) / d)[iu]


def composition(pos, num):
    return np.array([(num == z).sum() for z in (1, 6, 7, 8, 16)], float)


# The distance histogram needs an absolute length range, so the coordinates, which QM7 stores
# in bohr, are converted to angstrom first and the range is read from the data. A hard-coded
# 0-12 bohr window would both depend on the unit and silently drop every longer pair.
_iu15 = np.triu_indices(15, 1)
_max_pair_ang = max(
    np.linalg.norm((p * ANGSTROM_PER_BOHR)[:, None] - (p * ANGSTROM_PER_BOHR)[None, :], axis=-1)[_iu15].max()
    for p in positions
)
SHAPE_RANGE = (0.0, float(np.ceil(_max_pair_ang)))


def shape(pos, num):
    """Geometric shape descriptor: coordinate spread, second-moment eigenvalues, distance histogram.

    These are UNWEIGHTED geometric quantities, every atom counting equally, not the mass-weighted
    radius of gyration or moments of inertia, so they are named to say so. Coordinates arrive in
    bohr and are converted to angstrom, because the histogram range is an absolute length.
    """
    coords = pos * ANGSTROM_PER_BOHR
    centered = coords - coords.mean(0)
    geometric_spread = np.sqrt((centered ** 2).sum(1).mean())
    coord_moment_eigs = np.linalg.eigvalsh(centered.T @ centered)
    d = np.linalg.norm(coords[:, None] - coords[None, :], axis=-1)
    hist, _ = np.histogram(d[np.triu_indices(len(num), 1)], bins=12, range=SHAPE_RANGE, density=True)
    return np.concatenate([[geometric_spread], coord_moment_eigs, hist])


# For the record: how many pairs the old 0-12 bohr window silently dropped.
_dropped = _total = 0
for p in positions:
    d_bohr = np.linalg.norm(p[:, None] - p[None, :], axis=-1)[_iu15]
    _dropped += int((d_bohr > 12).sum()); _total += d_bohr.size
print(f"distance histogram range {SHAPE_RANGE} angstrom; the old 0-12 bohr window dropped "
      f"{_dropped}/{_total} pairs ({100 * _dropped / _total:.1f}%)")

REPS = {name: np.array([f(p, n) for p, n in zip(positions, numbers)])
        for name, f in [("Coulomb matrix", coulomb_offdiag),
                        ("Composition", composition),
                        ("Shape", shape)]}
for k, v in REPS.items():
    print(f"  {k:16s} {v.shape}")

### 6.1 Question one: is there structure at all?

Each representation is standardized and tested against a permutation null (Section 8.1).

In [ ]:
from sklearn.preprocessing import StandardScaler

SCALED = {k: StandardScaler().fit_transform(v) for k, v in REPS.items()}

def best_silhouette(X, seed=SEED):
    return max(silhouette_score(X, KMeans(k, n_init=10, random_state=seed).fit_predict(X))
               for k in range(2, 9))

for name, X in SCALED.items():
    observed = best_silhouette(X)
    r = np.random.default_rng(SEED)
    null = np.array([best_silhouette(permutation_null(X, r), SEED + i) for i in range(199)])
    p_value = (int(np.sum(null >= observed)) + 1) / (len(null) + 1)
    print(f"{name:16s} S = {observed:.3f}   null mean = {null.mean():.3f}   p = {p_value:.3f} (199 permutations)")
print()
print("All three say YES. If you stop here, you conclude the data is clustered, and you")
print("would be right. That is not the same as knowing what the clusters are.")

### 6.2 Question two: do they agree about *what* the clusters are?

This is the step almost nobody takes, and it is one function call.

### Give the adjusted Rand index a p-value before you read it

The adjusted Rand index counts the pairs of molecules two partitions agree
about, either together in both or apart in both, and corrects for the agreement
you would get by chance alone. One means the two partitions are identical, zero
means chance.

Corrected in expectation is not the same as having a sampling distribution, so
you cannot look at an ARI of 0.15 and call it significant. `partition_agreement`
supplies the missing piece: hold one label vector still, shuffle the other a few
hundred times, and see where the observed value falls among those. The
permutation route is valid for any configuration of cluster sizes.

`agreement_matrix` is the same index over every pair in a set of partitions,
which is how you compare three representations, or three algorithms, at once.

In [ ]:
from sklearn.metrics import adjusted_rand_score


def agreement_matrix(partitions):
    """Pairwise adjusted Rand index between named partitions of the same objects."""
    names = list(partitions)
    matrix = np.eye(len(names))
    for i, a in enumerate(names):
        for j in range(i + 1, len(names)):
            value = float(adjusted_rand_score(partitions[a], partitions[names[j]]))
            matrix[i, j] = matrix[j, i] = value
    return pd.DataFrame(matrix, index=names, columns=names)


def partition_agreement(labels_a, labels_b, n_permutations=999, random_state=0):
    """ARI between two partitions of the same objects, with a permutation p-value.

    Returns (observed, null_values, p_value). The smallest p-value it can report
    is 1 / (n_permutations + 1).
    """
    a, b = np.asarray(labels_a), np.asarray(labels_b)
    observed = float(adjusted_rand_score(a, b))

    gen = np.random.default_rng(random_state)
    permuted = b.copy()
    null = np.empty(n_permutations)
    for i in range(n_permutations):
        gen.shuffle(permuted)
        null[i] = adjusted_rand_score(a, permuted)

    p_value = (int(np.sum(null >= observed)) + 1) / (n_permutations + 1)
    return observed, null, p_value


def agreement_summary(observed, null, p_value):
    """Effect size first, p-value second: the ARI is the number that decides."""
    verdict = "agreement exceeds chance" if p_value < 0.05 else "NO better than chance"
    return (f"ARI = {observed:+.3f} (null {null.mean():+.3f}, "
            f"p = {p_value:.3f}) -> {verdict}")

In [ ]:
labels = {n: KMeans(4, n_init=10, random_state=SEED).fit_predict(X) for n, X in SCALED.items()}
print("Adjusted Rand index between representations:")
print(agreement_matrix(labels).round(3).to_string())
print()
ari, null, p = partition_agreement(labels["Coulomb matrix"], labels["Shape"], n_permutations=499)
print("Coulomb vs Shape:", agreement_summary(ari, null, p))

### Look at a representative of each cluster before you believe them

The guide argues that a clustering means nothing until it is interpreted with information the algorithm never had, and the most direct form of that is looking at the molecules. QM7 arrives as coordinates rather than SMILES, so the bonds are perceived from the geometry to draw them. If the four Coulomb-matrix clusters correspond to something a chemist would recognize, their representatives should differ in a way you can name; if they look interchangeable, that is a finding too, and it should temper any chemical story told about the partition.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, rdDetermineBonds
from IPython.display import display

BOHR = 0.529177210903


def atoms_to_mol(atoms):
    """RDKit molecule with perceived bonds from ASE atoms; QM7 ships no SMILES.

    Positions are converted from bohr to angstrom first, because bond perception
    reasons about interatomic distances and would find no bonds at bohr scale.
    Returns None when perception fails rather than raising.
    """
    pos = atoms.get_positions() * BOHR
    block = f"{len(atoms)}\n\n" + "\n".join(
        f"{s} {x:.5f} {y:.5f} {z:.5f}"
        for s, (x, y, z) in zip(atoms.get_chemical_symbols(), pos))
    mol = Chem.MolFromXYZBlock(block)
    if mol is None:
        return None
    try:
        rdDetermineBonds.DetermineBonds(mol, charge=0)
    except (ValueError, RuntimeError):
        return None
    return mol


# One representative per Coulomb-matrix cluster: the member nearest its centroid.
Xc = SCALED["Coulomb matrix"]
lab = labels["Coulomb matrix"]
medoids = []
for c in sorted(set(lab)):
    idx = np.where(lab == c)[0]
    centroid = Xc[idx].mean(axis=0)
    medoids.append(int(idx[np.argmin(((Xc[idx] - centroid) ** 2).sum(axis=1))]))

medoid_mols = [atoms_to_mol(fifteen[m]) for m in medoids]
keep = [i for i, m in enumerate(medoid_mols) if m is not None]
display(Draw.MolsToGridImage(
    [medoid_mols[i] for i in keep], molsPerRow=4, subImgSize=(260, 200),
    legends=[f"cluster {sorted(set(lab))[i]}" for i in keep]))

An ARI of 0.02-0.13 is close to chance. The molecules grouped together under one representation
are largely *not* the molecules grouped together under another.

Look at the permutation test above: Coulomb vs Shape agreement is *statistically* above chance
(p = 0.002) and *practically* negligible (ARI = 0.07). With 1219 molecules almost any
systematic effect is detectable. An effect size is how big a difference is, in units that do
not depend on how many molecules you have, and here the ARI is that number; the p-value says
only how confidently the difference was detected. Decide before the test how big a difference
would change what you do. That is the statistical-versus-practical distinction of Section 9 of
the article again, and it is why you report the effect size and not the p-value.

### 6.3 The algorithm changes the answer too


In [ ]:
from sklearn.cluster import AgglomerativeClustering

Xc = SCALED["Coulomb matrix"]
algos = {
    "k-means": KMeans(4, n_init=10, random_state=SEED).fit_predict(Xc),
    "Ward": AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(Xc),
    "average-linkage": AgglomerativeClustering(n_clusters=4, linkage="average").fit_predict(Xc),
}
print("Adjusted Rand index between algorithms, same representation:")
print(agreement_matrix(algos).round(3).to_string())

### 6.3b Does the disagreement survive a canonical atom ordering?

The comparison above used the Coulomb matrix in the file's given atom order, which Section 6.1
flags as a hazard. If the near-zero agreement were an artifact of that arbitrary order, sorting
the matrix canonically would remove it.

In [ ]:
# Item 9 robustness: does the near-zero cross-algorithm agreement depend on the file's atom
# order? Rebuild the Coulomb matrix in a canonical order (rows and columns sorted by row norm,
# which is permutation invariant) and repeat the comparison on it.
def coulomb_sorted(pos, num):
    z = num.astype(float)
    d = np.linalg.norm(pos[:, None] - pos[None, :], axis=-1)
    np.fill_diagonal(d, np.inf)
    M = np.outer(z, z) / d
    np.fill_diagonal(M, 0.5 * z ** 2.4)
    order = np.argsort(-np.linalg.norm(M, axis=1))
    M = M[np.ix_(order, order)]
    return M[np.triu_indices(len(z), 1)]


Xc_sorted = StandardScaler().fit_transform(
    np.array([coulomb_sorted(p, n) for p, n in zip(positions, numbers)]))
algos_sorted = {
    "k-means": KMeans(4, n_init=10, random_state=SEED).fit_predict(Xc_sorted),
    "Ward": AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(Xc_sorted),
    "average": AgglomerativeClustering(n_clusters=4, linkage="average").fit_predict(Xc_sorted),
}
print("Adjusted Rand index between algorithms, canonically (row-norm) sorted Coulomb matrix:")
print(agreement_matrix(algos_sorted).round(3).to_string())
print()
print("Cross-algorithm agreement stays low under the canonical ordering, so the disagreement is")
print("a property of the data and the algorithms, not an artifact of the file's atom order; the")
print("Section 8.6 conclusion does not need a re-run on sorted descriptors.")

Here is how to read that matrix. Each entry is an adjusted Rand index between two partitions of the same 1,219 molecules, so 1 means the two algorithms grouped the molecules identically, 0 means they agree no better than chance, and anything below about 0.2 means they are telling you different stories. The diagonal is 1 by construction and carries no information.

Compare this against the previous section. There the *representation* was varied and the algorithm held fixed; here the algorithm varies and the representation is fixed. If both sets of numbers are low, neither choice is incidental, and reporting one clustering without saying which algorithm and which representation produced it is reporting one draw from a set of possible answers.


### 6.4 Check why a silhouette is high before you believe it

The Composition representation scores a silhouette around 0.95, far above the others. It is
tempting to conclude that it is the best representation, so check where that number comes from
before you believe it.

In [ ]:
comp = REPS["Composition"]
unique_rows = len(np.unique(comp, axis=0))
print(f"{len(comp)} molecules occupy only {unique_rows} distinct composition vectors")
print(f"i.e. on average {len(comp) / unique_rows:.0f} molecules sit at exactly the same point.")
print()
print("The clusters are tight because the representation is DISCRETE, not because the")
print("chemistry is clean. Silhouette measures compactness; degeneracy produces compactness")
print("for free. This is why an index must be read alongside what the representation is.")

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

from sklearn.metrics import adjusted_rand_score

# The article quotes the BEST silhouette over k, which is what Section 6.1
# reported and what makes the number look impressive. Scoring a fixed k=4 here
# instead would guard a different quantity and let the quoted one drift freely.
sil_comp = best_silhouette(SCALED["Composition"])
rep_pairs = [adjusted_rand_score(labels[a], labels[b])
             for i, a in enumerate(labels) for b in list(labels)[i + 1:]]
alg_pairs = [adjusted_rand_score(algos[a], algos[b])
             for i, a in enumerate(algos) for b in list(algos)[i + 1:]]

print(f"silhouette, composition            {sil_comp:8.2f}   article: 0.95")
print(f"ARI between representations        {min(rep_pairs):8.2f} to {max(rep_pairs):.2f}   article: 0.02-0.13")
print(f"ARI between algorithms (Coulomb)   {min(alg_pairs):8.2f} to {max(alg_pairs):.2f}   article: ~0.00")
print(f"distinct composition vectors       {unique_rows:8d} of {len(comp)}")

assert sil_comp > 0.85, f"composition silhouette drifted: {sil_comp:.3f}"
assert max(rep_pairs) < 0.35, f"representations now agree at ARI {max(rep_pairs):.2f}; the case study assumes they do not"
assert min(alg_pairs) < 0.20, f"algorithms now agree at ARI {min(alg_pairs):.2f}"
assert unique_rows < len(comp) / 2, "composition is no longer degenerate; the section 2 argument depends on it"
print("\nall within tolerance of the values printed in the article")

---
## 7. Choose the fingerprint and the metric deliberately

Everything from here uses RDKit, which comes with the core install. These cells answer the practical questions from Section 8.5 of the article, "Clustering molecules by fingerprint", and extend the Section 8.6 case study above to fingerprints.

### Standardize before you fingerprint

The ZINC SMILES arrive as distributed: salt forms, charged forms, and whatever
convention the source happened to use. Two records of one compound written two
ways land in different regions of fingerprint space, and an unsupervised method
will report that as a chemical distinction.

The pipeline below is the short version of notebook 04. Parse the string, keep
the largest fragment (which drops counter-ions), neutralize what the fragment
removal left charged, and write a canonical SMILES. Each step is a chemical
judgment, so say in your methods which ones you applied.

In [ ]:
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")   # parse failures are handled below, not printed

# Build the RDKit helpers once. Constructing them per molecule dominates the
# runtime on anything larger than a toy set.
_largest_fragment = rdMolStandardize.LargestFragmentChooser()
_uncharger = rdMolStandardize.Uncharger()


def standardize(smiles):
    """Largest fragment, neutralized, canonical SMILES. None if it will not parse."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    if "." in smiles:                      # a salt, or two species written together
        mol = _largest_fragment.choose(mol)
    mol = _uncharger.uncharge(mol)
    return Chem.MolToSmiles(mol, canonical=True)

In [ ]:
# The shared sample carries 10,000 records because notebook 04 needs them. The
# Taylor-Butina and agglomerative clustering below build a full pairwise distance
# matrix, whose cost is quadratic, so take the first 1,000 rows.
zinc = pd.read_csv(DATA / "zinc-250k-sample.csv").head(1000)
smiles = sorted({s for s in (standardize(x) for x in zinc["smiles"].str.strip()) if s is not None})
mols = [Chem.MolFromSmiles(s) for s in smiles]
print(f"{len(mols)} standardized molecules")

morgan_bit = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)
FPS = {
    "ECFP4 bit (2048)": np.array([morgan_bit.GetFingerprintAsNumPy(m) for m in mols], float),
    "ECFP4 count (2048)": np.array([morgan_bit.GetCountFingerprintAsNumPy(m) for m in mols], float),
    "ECFP4 bit (1024)": np.array([
        rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=True)
        .GetFingerprintAsNumPy(m) for m in mols], float),
    "MACCS (166)": np.array([np.array(MACCSkeys.GenMACCSKeys(m)) for m in mols], float),
}
for k, v in FPS.items():
    density = (v > 0).mean()
    print(f"  {k:20s} {v.shape}, mean bit density {density:.3f}")

In [ ]:
from rdkit.Chem import Draw
from IPython.display import display

# Eight of the ZINC molecules the fingerprint comparison runs on.
display(Draw.MolsToGridImage(mols[:8], molsPerRow=4, subImgSize=(260, 200)))

### 7.1 Tanimoto similarity depends on molecular size

This is the oldest known pitfall, and the one that most often produces clusters organized by
size instead of by chemistry.

In [ ]:
from rdkit.Chem.Descriptors import HeavyAtomCount
from scipy.stats import spearmanr

heavy = np.array([HeavyAtomCount(m) for m in mols])
fps_bit = [morgan_bit.GetFingerprint(m) for m in mols]

# For each molecule, its mean Tanimoto similarity to all others.
# Each molecule's similarity to itself is 1.0; exclude it so the mean is over the OTHERS.
mean_sim = np.array([
    (np.sum(DataStructs.BulkTanimotoSimilarity(fp, fps_bit)) - 1.0) / (len(fps_bit) - 1)
    for fp in fps_bit
])
rho, p = spearmanr(heavy, mean_sim)
print(f"Spearman correlation between heavy-atom count and mean Tanimoto similarity:")
print(f"  rho = {rho:+.3f}  (p = {p:.2e})")
print()
print("If this is strongly non-zero, then a cluster of similar molecules is partly a cluster")
print("of similarly-sized molecules. Always check cluster membership against size.")

fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(heavy, mean_sim, s=6, alpha=0.4, linewidths=0)
ax.set_xlabel("heavy atom count"); ax.set_ylabel("mean Tanimoto to all others")
ax.set_title(f"Size dependence of Tanimoto similarity (rho = {rho:+.2f})")
plt.show()

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

print(f"Spearman rho, size vs similarity   {rho:+8.2f}")
assert abs(rho) > 0.25, f"the Tanimoto size bias has vanished (rho = {rho:+.3f}); Section 8.5 depends on it"
print("within tolerance")

### 7.2 Do bit, count, and MACCS fingerprints agree about the clusters?

This is the same test as Section 6, now run over fingerprint choices. Watch the metric:
Tanimoto/Jaccard for binary fingerprints, **not** Euclidean.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def cluster_fp(matrix, k=6, binary=True):
    metric = "jaccard" if binary else "cosine"
    return AgglomerativeClustering(n_clusters=k, metric=metric,
                                   linkage="average").fit_predict(matrix)

fp_labels = {
    "ECFP4 bit (2048)": cluster_fp(FPS["ECFP4 bit (2048)"]),
    "ECFP4 bit (1024)": cluster_fp(FPS["ECFP4 bit (1024)"]),
    "ECFP4 count (2048)": cluster_fp(FPS["ECFP4 count (2048)"], binary=False),
    "MACCS (166)": cluster_fp(FPS["MACCS (166)"]),
}
print("Adjusted Rand index between fingerprint choices:")
print(agreement_matrix(fp_labels).round(3).to_string())
print()
print("Read the ECFP-2048 vs ECFP-1024 entry carefully: that is the SAME fingerprint at two")
print("folding lengths: a parameter most people never report.")

### 7.3 MACCS similarity values live on a different scale

A Tanimoto threshold of 0.7 does not mean the same thing for MACCS as for ECFP4. This is why
similarity cut-offs are not transferable between fingerprint types.

In [ ]:
maccs_fps = [MACCSkeys.GenMACCSKeys(m) for m in mols]
idx = rng.choice(len(mols), size=400, replace=False)

def pairwise_sample(fps, index):
    out = []
    for i in index[:200]:
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], [fps[j] for j in index[200:]])
        out.extend(sims)
    return np.array(out)

sim_ecfp = pairwise_sample(fps_bit, idx)
sim_maccs = pairwise_sample(maccs_fps, idx)

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.hist(sim_ecfp, bins=50, alpha=0.6, density=True, label=f"ECFP4 (median {np.median(sim_ecfp):.2f})")
ax.hist(sim_maccs, bins=50, alpha=0.6, density=True, label=f"MACCS (median {np.median(sim_maccs):.2f})")
ax.axvline(0.7, color="gray", ls="--", lw=1)
ax.text(0.71, ax.get_ylim()[1] * 0.9, "T = 0.7", fontsize=7, color="gray")
ax.set_xlabel("Tanimoto similarity between random pairs"); ax.set_ylabel("density")
ax.set_title("The same threshold means different things")
ax.legend(frameon=False)
plt.show()

### 7.4 Measure what the wrong metric costs you

Euclidean distance on binary fingerprints is not defensible on its own (Bajusz et al. 2015).
Here is how much it changes the partition.

In [ ]:
euclid_labels = AgglomerativeClustering(n_clusters=6, metric="euclidean",
                                        linkage="ward").fit_predict(FPS["ECFP4 bit (2048)"])
ari, null, p = partition_agreement(fp_labels["ECFP4 bit (2048)"], euclid_labels, n_permutations=499)
print("Jaccard-based vs Euclidean-based clustering of the SAME fingerprints:")
print("  " + agreement_summary(ari, null, p))

### 7.5 Reach for Taylor-Butina on fingerprints, and scan its one dial

Butina is the algorithm to reach for when the objects *are* fingerprints. It works directly on
Tanimoto distances, assumes nothing about cluster shape or size, and is allowed to leave a
molecule on its own. Reach for k-means or HDBSCAN instead when the representation is
continuous (learned embeddings, physicochemical descriptors, geometric features), where
Tanimoto is not defined at all.

Its output is then governed almost entirely by one number, the Tanimoto cutoff, and that number
is usually inherited from a previous paper, not chosen. Give it the same treatment Section 5
gives DBSCAN's `eps`: scan it, and report the **singleton fraction** (Butina's version of
DBSCAN's noise fraction) alongside the cluster count.

### Read Taylor-Butina in one function

Taylor-Butina is short enough to state in prose: compute every pairwise
Tanimoto distance, count for each molecule how many neighbors fall inside the
cutoff, then take the molecule with the most neighbors, call it a cluster
center, and remove it and its neighbors from the pool. Repeat that on what is
left, and anything that ends up alone stays alone as a singleton.

That makes it a leader algorithm, and two consequences follow, both of them
measured below: the partition depends on the cutoff, and it depends on the order
the molecules arrive in when neighbor counts tie. RDKit's `Butina.ClusterData`
wants the distances as a flat lower-triangle list, which is what the loop
builds.

In [ ]:
from rdkit.ML.Cluster import Butina


def butina_clusters(smiles_list, tanimoto_cutoff=0.65, radius=2, n_bits=2048):
    """Taylor-Butina cluster index for each molecule.

    Butina.ClusterData takes a DISTANCE threshold, so a Tanimoto cutoff of 0.65
    (cluster molecules at least that similar) is passed as the distance
    1 - 0.65 = 0.35. Passing the similarity directly, as an earlier version did,
    clustered at Tanimoto 0.35 while the label said 0.65.
    """
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius, fpSize=n_bits, includeChirality=True
    )
    fps = [generator.GetFingerprint(Chem.MolFromSmiles(s)) for s in smiles_list]

    # The lower triangle of the distance matrix, flattened row by row.
    distances = []
    for i in range(1, len(fps)):
        similarities = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        distances.extend(1.0 - s for s in similarities)

    clusters = Butina.ClusterData(distances, len(fps), 1.0 - tanimoto_cutoff, isDistData=True)
    labels = np.empty(len(smiles_list), dtype=int)
    for cluster_id, members in enumerate(clusters):
        for m in members:
            labels[m] = cluster_id
    return labels

In [ ]:
def butina_summary(labels):
    _, counts = np.unique(labels, return_counts=True)
    return len(counts), (counts == 1).sum() / len(labels), counts.max() / len(labels)

butina_labels = {c: butina_clusters(smiles, tanimoto_cutoff=c) for c in (0.8, 0.65, 0.5, 0.35, 0.2)}

print(f"{'Tanimoto':>8} {'clusters':>9} {'in singletons':>14} {'largest cluster':>16}")
for tcut, labels in butina_labels.items():
    n_clusters, singleton_frac, largest_frac = butina_summary(labels)
    print(f"{tcut:8.2f} {n_clusters:9d} {singleton_frac:14.2f} {largest_frac:16.2f}")
print()
print("Read down the two right-hand columns. If they move steadily with the cutoff, with no")
print("plateau where the partition holds still, then the number you picked IS the result,")
print("so record it next to the fingerprint and its length.")

In [ ]:
# Butina is a leader algorithm: the first molecule in a neighborhood claims it. So the
# partition depends on the order the molecules arrive in, which almost nobody reports.
order = np.random.default_rng(SEED).permutation(len(smiles))
shuffled = butina_clusters([smiles[i] for i in order], tanimoto_cutoff=0.65)
restored = np.empty_like(shuffled)
restored[order] = shuffled

print("Same molecules, same fingerprint, same cutoff, shuffled input order:")
print(f"  ARI = {adjusted_rand_score(butina_labels[0.65], restored):.3f}")
print("Fix the input order and record it, or the partition is not reproducible.")
print()

# And how much does the choice of algorithm matter, holding the fingerprint fixed?
ari, null, p = partition_agreement(butina_labels[0.65], fp_labels["ECFP4 bit (2048)"],
                                   n_permutations=499)
print("Butina (T = 0.65) vs average-linkage Jaccard at k = 6, on the SAME fingerprints:")
print("  " + agreement_summary(ari, null, p))

### 7.6 Scaffolds stop carrying signal as molecules get bigger

Bemis-Murcko scaffolds are the usual grouping key for splitting, and a tempting one for
clustering. The framework was defined on drug-sized molecules, where stripping the side chains
leaves a recognizable core that carries most of the chemical meaning. That assumption fails on
large modular molecules. In a PROTAC the warhead, the linker and the E3 ligand are all ring
systems joined by chains, so nearly every heavy atom survives into the "scaffold", and the
same holds for molecular glues, macrocycles and peptidomimetics.

The diagnostic is a ratio: how many of a molecule's heavy atoms end up in its scaffold. When
that approaches one, the scaffold is the molecule, every compound gets its own group, and a
scaffold split or a scaffold-based clustering is a random partition wearing a chemical
justification. This is the same pathology as the degeneracy check in the Representations
checklist, seen from the other side: there, different molecules got the *same* vector; here,
near-identical molecules get *different* groups.

### Compute Murcko scaffolds, and decide what to do with acyclic molecules

A Bemis-Murcko scaffold is what survives when you strip the side chains: the
ring systems plus the linkers that join them. RDKit computes it directly, so most of
this function is bookkeeping. The one case that needs a decision is an acyclic
molecule, which has no rings and so an empty scaffold. Pooling those under a
single empty key would make every acyclic molecule one group, and in a split
that group would straddle both sides, so each gets its own placeholder instead.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold


def bemis_murcko_scaffolds(smiles_list):
    """Bemis-Murcko scaffold for each molecule, as a SMILES string."""
    scaffolds = []
    for i, smi in enumerate(smiles_list):
        scaffold = MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(smi))
        key = Chem.MolToSmiles(scaffold)
        scaffolds.append(key if key else f"__acyclic_{i}__")
    return scaffolds

In [ ]:
def scaffold_coverage(smiles_list):
    """Fraction of each molecule's heavy atoms that survive into its Murcko scaffold."""
    out = []
    for smi, scaffold in zip(smiles_list, bemis_murcko_scaffolds(smiles_list)):
        mol, core = Chem.MolFromSmiles(smi), Chem.MolFromSmiles(scaffold)
        # bemis_murcko_scaffolds returns a placeholder for acyclic molecules: no scaffold at all.
        out.append(0.0 if core is None else HeavyAtomCount(core) / HeavyAtomCount(mol))
    return np.array(out)


# Real targeted degraders, not synthetic constructs: the 1000 TPDdb PROTACs the repository ships.
from collections import Counter

_protac_df = pd.read_csv(DATA / "protac-tpddb-sample.csv")
protac_smiles = [standardize(s) for s in _protac_df["smiles"].str.strip()]
protac_smiles = [s for s in protac_smiles if s is not None]

cov = scaffold_coverage(smiles)                  # ZINC drug-sized sample, kept for comparison
cov_protac = scaffold_coverage(protac_smiles)    # real PROTACs
protac_scaffolds = bemis_murcko_scaffolds(protac_smiles)
per_scaffold = np.array([n for _, n in Counter(protac_scaffolds).items()])

print(f"ZINC drug-sized sample   median scaffold coverage {np.median(cov):.2f}")
print(f"real PROTAC sample       median scaffold coverage {np.median(cov_protac):.2f}, "
      f"{100 * (cov_protac > 0.8).mean():.0f}% above 0.8")
print(f"                         {len(set(protac_scaffolds))} distinct scaffolds for "
      f"{len(protac_smiles)} molecules, {int((per_scaffold == 1).sum())} of them singletons")
print()
print("A high median coverage means the scaffold is nearly the whole molecule, so grouping on it")
print("gives almost one group per compound. Coverage is a diagnostic and not a rule, though:")
for label, smi in [("fused polycyclic (coronene)", "c1cc2ccc3ccc4ccc5ccc6ccc1c1c2c3c4c5c61"),
                   ("linear tripeptide (Gly-Gly-Gly)", "NCC(=O)NCC(=O)NCC(=O)O")]:
    print(f"  {label:32s} coverage {scaffold_coverage([smi])[0]:.2f}")
print("the fused ring scores high with a scaffold that IS the chemistry, the peptide low with a")
print("tiny scaffold despite its size, so read coverage alongside what the scaffold actually is.")

In [ ]:
heavy_protac = np.array([HeavyAtomCount(Chem.MolFromSmiles(s)) for s in protac_smiles])
rho_cov, _ = spearmanr(heavy, cov)  # 'heavy' and spearmanr come from Section 7.1

fig, ax = plt.subplots(figsize=(5.2, 3))
ax.scatter(heavy, cov, s=6, alpha=0.35, linewidths=0,
           label=f"ZINC sample (rho = {rho_cov:+.2f})")
ax.scatter(heavy_protac, cov_protac, s=34, color="crimson", marker="^",
           label="real PROTACs")
ax.axhline(0.8, color="gray", ls="--", lw=1)
ax.set_xlabel("heavy atom count")
ax.set_ylabel("scaffold atoms / molecule atoms")
ax.set_title("The scaffold stops being a summary as molecules grow")
ax.legend(frameon=False, fontsize=7, loc="lower right")
plt.show()

print("Rule of thumb: before using scaffolds as a grouping key, report the median coverage and")
print("the number of molecules per scaffold. Above roughly 0.8, or when most scaffolds hold a")
print("single molecule, group on something that does carry signal instead: the shared warhead")
print("or E3 ligand substructure, the linker class, or a fingerprint distance (Section 7.5),")
print("and say which one you used.")

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# The article quotes PROTAC scaffold coverage "above roughly 0.8"; this checks the real number.
print(f"median coverage, ZINC vs PROTAC   {np.median(cov):.2f} vs {np.median(cov_protac):.2f}   article: PROTAC above ~0.8")
assert np.median(cov_protac) > 0.8, f"real PROTAC median coverage should be above 0.8, got {np.median(cov_protac):.2f}"
assert np.median(cov_protac) > np.median(cov), "PROTAC coverage should exceed the drug-sized sample's"
assert (per_scaffold == 1).mean() > 0.5, "most PROTAC scaffolds should be singletons (one molecule each)"
print("within tolerance")

## Manuscript figures


- **This is Figure 6 of the manuscript** (`silhouette_selection_bias`): silhouette scans against permutation null bands on structureless data and on data with real clusters, with bootstrap stability alongside.
- **This is Figure 7 of the manuscript** (`cluster1_angle`): the aspirin trajectory clusters checked against a ring-carboxyl dihedral the clustering never saw.
- **This is Figure 8 of the manuscript** (`clustering_illusion_molecules`): the same 1219 QM7 molecules under three representations, and the adjusted Rand agreement between representations and between algorithms.


This cell produces the article's Section 8 figures from the code that generated the published
versions: `silhouette_selection_bias` (**Figure 6**, the silhouette scans against their
permutation null bands, with the bootstrap-stability comparison as its third panel),
`cluster1_angle` (**Figure 7**, the aspirin clusters checked against a dihedral angle the
clustering never saw), and `clustering_illusion_molecules` (**Figure 8**, the manuscript
version of Section 6 above: PCA views of the same 15-atom QM7 molecules under three
representations with their observed silhouette and its distance above a permutation null,
the adjusted-Rand agreement between representations and between algorithms, and the reading
of those numbers). Every figure is written as **both a PDF and a PNG**.

The cell reuses the functions built earlier in the notebook and adds only what the printed
page needs: figure widths measured from the compiled document, smaller type, and panel
letters. Those letters are `ax.text` and not titles: they say *which* panel you are looking
at, where a title would state a claim, and a published figure states its claim in the caption. No
plot in this cell sets a title. The last line restores the notebook style, so anything you add
below gets screen-sized type again.

In [ ]:
# --- Manuscript figures: silhouette_selection_bias, cluster1_angle,
# --- clustering_illusion_molecules (Section 8; Figure 8 is the last).
#
# The figure code lives here rather than in a script, so that the notebook a
# reader follows and the figure the article prints cannot drift apart.

from ase.io import read
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


import os


def savefig(fig, stem, outdir=None):
    """Write a figure as both a PNG and a PDF to a local ``figures/`` directory.

    Set the FIGDIR environment variable, or pass outdir, to write somewhere else.
    """
    outdir = Path(outdir or os.environ.get("FIGDIR") or "figures")
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in ("png", "pdf"):
        fig.savefig(outdir / f"{stem}.{fmt}")
    return outdir


# Type sized for the printed column rather than the screen. 7.5 pt at the figure
# widths below is 7.5 pt on the page, because nothing rescales afterwards.
set_style()
mpl.rcParams.update(
    {
        "figure.dpi": 200,
        "font.size": 7.5,
        "axes.labelsize": 7.5,
        "axes.titlesize": 8,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 2.8,
        "ytick.major.size": 2.8,
        "lines.linewidth": 1.3,
    }
)

# Canonical figure widths, in inches, measured from the compiled document:
#   \linewidth = 250.95 pt = 3.47 in  (one column of the two-column layout)
#   \textwidth = 520.40 pt = 7.20 in  (both columns, i.e. a figure* float)
COL_W = 3.47
FULL_W = 7.20

# Panel letters: one size and offset for the whole article, so that (a) in one
# figure looks like (a) in another.
PANEL_SIZE = 8.5
PANEL_XY = (-0.09, 1.01)
MARKERS = ["o", "s", "^", "D"]

# One sequential colormap for every magnitude in the article: near-white to
# purple, one hue, monotone in lightness. A rainbow map varies in hue as well,
# which invites the reader to see categories inside a continuum.
HEATMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "manuscript_heatmap", ["#F6F2FA", LAVENDER, PURPLE]
)

# SEED is the one set at the top of this notebook (20260726), the same value the
# figures were originally generated with.


def add_panel_label(ax, label, x=PANEL_XY[0], y=PANEL_XY[1], size=PANEL_SIZE):
    """The bold (a), (b), (c), placed outside the axes. Text, never a title."""
    ax.text(x, y, label, transform=ax.transAxes, fontsize=size, fontweight="bold",
            va="bottom", ha="left", color=SLATE)


def panel_caption(ax, text, size=7.0):
    """Which panel this is. What it means goes in the figure caption."""
    ax.text(0.5, 1.012, text, transform=ax.transAxes, ha="center", va="bottom",
            fontsize=size, color=SLATE)


def cluster_style(index):
    """Color and marker for one group of a four-group scatter.

    Every categorical distinction carries a marker shape as well as a color, so
    the panels survive a grayscale print.
    """
    i = index % len(PALETTE)
    return PALETTE[i], MARKERS[i]


def null_data(n=800, d=50, seed=SEED):
    """Isotropic Gaussian noise: a cloud with no cluster structure whatsoever."""
    return np.random.default_rng(seed).standard_normal((n, d))


def clustered_data(n_per=200, d=50, k=4, sep=3.2, seed=SEED):
    """Genuine, well-separated Gaussian clusters in the first two dimensions."""
    gen = np.random.default_rng(seed)
    centers = np.zeros((k, d))
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    centers[:, 0] = sep * np.cos(angles)
    centers[:, 1] = sep * np.sin(angles)
    X = np.vstack([gen.standard_normal((n_per, d)) + c for c in centers])
    return X, np.repeat(np.arange(k), n_per)


def silhouette_over_k(X, ks, seed):
    """Mean silhouette of a k-means fit at each k in ks."""
    return np.array([
        silhouette_score(X, KMeans(n_clusters=int(k), n_init=10,
                                   random_state=seed).fit_predict(X))
        for k in ks
    ])


def fig_silhouette_selection_bias(n_null=199, annotate=True):
    ks = np.arange(2, 13)

    X_null = null_data(n=600, d=20)
    X_real, _ = clustered_data(n_per=150, d=20, k=4)

    s_null = silhouette_over_k(X_null, ks, SEED)
    s_real = silhouette_over_k(X_real, ks, SEED)

    def null_band(X):
        """The same permutation null as Section 1, once per k."""
        gen = np.random.default_rng(SEED)
        band = np.zeros((n_null, len(ks)))
        for i in range(n_null):
            band[i] = silhouette_over_k(permutation_null(X, gen), ks, SEED + i)
        return np.percentile(band, [2.5, 97.5], axis=0)

    # Three panels: the two null-band scans, and the stability box plot that
    # was a separate figure in version 3. They belong together: (a) and (b)
    # ask "is there structure?", (c) asks "are the groups reproducible?", and
    # combining them removes a page of whitespace.
    # Equal-width panels for the clean variant so a, b and c read as one set;
    # the annotated version keeps the narrower stability panel it was tuned with.
    width_ratios = [1.0, 1.0, 0.85] if annotate else [1.0, 1.0, 1.0]
    wspace = 0.30 if annotate else 0.42
    # The clean variant is a little shorter, since it has no titles or arrows
    # above the panels to make room for.
    fig_h = 2.35 if annotate else 2.0
    fig, axes = plt.subplots(
        1, 3, figsize=(FULL_W, fig_h),
        gridspec_kw={"width_ratios": width_ratios, "wspace": wspace},
    )
    axes[1].sharey(axes[0])

    for ax, (X, s, color, marker, ls, caption, ktrue) in zip(
        axes[:2],
        [
            (X_null, s_null, PLUM, "D", (0, (4, 1.5)),
             "Structureless data\n(no clusters exist)", None),
            (X_real, s_real, GREEN, "o", "-",
             "Genuinely clustered data\n(four Gaussian clusters)", 4),
        ],
    ):
        lo, hi = null_band(X)
        ax.fill_between(
            ks, lo, hi, facecolor="none", edgecolor=SLATE, hatch="///", lw=0.4,
            alpha=0.8,
            label=(f"95% permutation null ({n_null} shuffles)" if annotate
                   else "95% permutation null"),
        )
        ax.plot(ks, s, color=color, marker=marker, linestyle=ls, ms=4,
                label="observed silhouette")
        if annotate:
            panel_caption(ax, caption)
        ax.set_xlabel("number of clusters $k$")

        if annotate and ktrue is None:
            kbest = ks[int(np.argmax(s))]
            ax.annotate(
                f"argmax at $k={kbest}$,\nbut inside the null band:\nno structure to find",
                xy=(kbest, s.max()),
                xytext=(kbest + 1.0, s.max() + 0.055),
                fontsize=6.2,
                arrowprops=dict(arrowstyle="->", lw=0.7, color=SLATE),
            )
        elif annotate:
            ax.annotate(
                f"true $k={ktrue}$;\nclearly above the null",
                xy=(ktrue, s[ktrue - ks[0]]),
                xytext=(ktrue + 2.0, s.max() - 0.035),
                fontsize=6.2,
                arrowprops=dict(arrowstyle="->", lw=0.7, color=SLATE),
            )

    axes[0].set_ylabel("mean silhouette")
    if annotate:
        # a and b share the scale and sit flush, so b's duplicate y labels are
        # hidden. The clean variant spaces the panels out and labels both.
        axes[1].tick_params(labelleft=False)
    else:
        axes[1].tick_params(labelleft=True)
        axes[1].set_ylabel("mean silhouette")
    # Panel (a) is flat and empty in its upper half; panel (b)'s lower-left is
    # occupied by the null band, so the legend goes where there is room.
    axes[0].legend(loc="upper right", fontsize=6.2, borderaxespad=0.3)

    stability_panel(axes[2], annotate=annotate)

    for ax, lab in zip(axes, ["(a)", "(b)", "(c)"]):
        add_panel_label(ax, lab)
    stem = ("silhouette_selection_bias" if annotate
            else "silhouette_selection_bias_no_annotations")
    outdir = savefig(fig, stem)
    print(f"wrote {stem}.png/.pdf to {outdir}")


def stability_panel(ax, annotate=True):
    X_real, _ = clustered_data(n_per=120, d=20, k=4)
    X_null = null_data(n=480, d=20)

    def kmeans_four(data):
        return KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(data)

    real_scores, _ = cluster_stability(X_real, kmeans_four, n_bootstrap=60, random_state=SEED)
    null_scores, _ = cluster_stability(X_null, kmeans_four, n_bootstrap=60, random_state=SEED)

    positions = [1, 2]
    parts = ax.boxplot(
        [real_scores.ravel(), null_scores.ravel()],
        positions=positions,
        widths=0.5,
        patch_artist=True,
        medianprops=dict(color=SLATE, lw=1.4),
        flierprops=dict(
            marker="o", ms=1.8, markerfacecolor=SLATE,
            markeredgecolor="none", alpha=0.4,
        ),
    )
    # Hatch as well as color: the distinction has to survive a grayscale print.
    for patch, color, hatch in zip(parts["boxes"], [GREEN, PLUM], ["", "xxx"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.50)
        patch.set_edgecolor(SLATE)
        patch.set_hatch(hatch)
        patch.set_linewidth(0.8)

    if annotate:
        ax.axhline(0.75, color=SLATE, ls="--", lw=0.9)
        ax.text(
            0.5, 0.765, "conventional threshold",
            transform=ax.transAxes, fontsize=5.8, color=SLATE,
            va="bottom", ha="center",
        )
    ax.set_xticks(positions)
    ax.set_xticklabels(["four real\nclusters", "no structure\n(noise)"], fontsize=6.4)
    ax.set_ylabel("bootstrap Jaccard stability", fontsize=7)
    ax.set_ylim(0, 1.04)


def fig_aspirin_dihedral():
    """External validation: cluster the trajectory, then check an order
    parameter the clustering was never given."""
    d = np.load(DATA / "md17_aspirin_10000.npz")
    R = d["R"]

    # Cluster on internal pairwise distances, which are invariant to rotation
    # and translation: the alignment-free option recommended in Section 5.
    iu = np.triu_indices(R.shape[1], 1)
    feats = np.linalg.norm(R[:, :, None, :] - R[:, None, :, :], axis=-1)[:, iu[0], iu[1]]
    labels = KMeans(2, n_init=10, random_state=SEED).fit_predict(feats)

    # ASPIRIN_DIHEDRAL and dihedral() come from Section 4. Signed, not absolute:
    # taking |angle| folds 0 and 180 degrees onto each other and hides the very
    # distinction the clusters encode.
    i, j, k, m = ASPIRIN_DIHEDRAL
    angles = dihedral(R[:, i], R[:, j], R[:, k], R[:, m])

    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.72))
    for cl, color, hatch in zip((0, 1), (GREEN, PURPLE), ("", "///")):
        sel = labels == cl
        ax.hist(
            angles[sel], bins=60, range=(-180, 180),
            color=color, alpha=0.55, edgecolor=SLATE, linewidth=0.5,
            hatch=hatch, label=f"cluster {cl} ({100 * sel.mean():.0f}% of frames)",
            density=False,
        )
    ax.set_xlabel("ring-carboxyl dihedral angle (degrees)")
    ax.set_ylabel("number of frames")
    ax.set_xlim(-180, 180)
    ax.set_xticks([-180, -90, 0, 90, 180])
    # Headroom above the peaks so the upper-center legend clears the bars.
    ax.set_ylim(0, ax.get_ylim()[1] * 1.35)
    ax.legend(loc="upper center", fontsize=6.2)
    outdir = savefig(fig, "cluster1_angle")
    print(f"wrote cluster1_angle.png/.pdf to {outdir}")

    energies = d["E"].ravel()
    for cl in (0, 1):
        sel = labels == cl
        # Circular mean, not median: one cluster sits on the anti conformation
        # at 180 degrees, whose samples straddle the +/-180 wrap, and an
        # ordinary median of those lands wherever the two lobes balance.
        radians = np.deg2rad(angles[sel])
        center = np.degrees(np.arctan2(np.sin(radians).mean(), np.cos(radians).mean()))
        print(f"   cluster {cl}: {sel.sum():5d} frames ({100 * sel.mean():.0f}%), "
              f"circular mean dihedral = {center:7.1f} deg, "
              f"mean energy offset = {energies[sel].mean() - energies.mean():+.2f}")


def fig_clustering_illusion_molecules(n_null=199):
    """Same molecules, three defensible representations, three different answers."""
    molecules = [a for a in read(str(DATA / "qm7.xyz"), index=":") if len(a) == 15]
    pos = np.array([a.get_positions() for a in molecules])
    num = np.array([a.get_atomic_numbers() for a in molecules])

    # The three representation functions are the ones from Section 6, so the
    # figure and the section above cannot disagree.
    reps = {
        "Coulomb matrix": np.array([coulomb_offdiag(p, n) for p, n in zip(pos, num)]),
        "Composition": np.array([composition(p, n) for p, n in zip(pos, num)]),
        "Shape": np.array([shape(p, n) for p, n in zip(pos, num)]),
    }
    scaled = {k: StandardScaler().fit_transform(v) for k, v in reps.items()}

    against_null = {}
    for name, X in scaled.items():
        observed = best_silhouette(X)
        gen = np.random.default_rng(SEED)
        null = np.array(
            [best_silhouette(permutation_null(X, gen), SEED + i) for i in range(n_null)]
        )
        against_null[name] = (observed, null)

    labels = {n: KMeans(4, n_init=10, random_state=SEED).fit_predict(X) for n, X in scaled.items()}
    algos = {
        "k-means": KMeans(4, n_init=10, random_state=SEED).fit_predict(scaled["Coulomb matrix"]),
        "Ward": AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(
            scaled["Coulomb matrix"]),
        "average": AgglomerativeClustering(n_clusters=4, linkage="average").fit_predict(
            scaled["Coulomb matrix"]),
    }
    rep_ari = agreement_matrix(labels)
    alg_ari = agreement_matrix(algos)

    fig = plt.figure(figsize=(FULL_W, 3.8))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.15, 1.0], hspace=0.28, wspace=0.34)

    panel_letters = iter(["(a)", "(b)", "(c)", "(d)", "(e)"])
    for col, (name, X) in enumerate(scaled.items()):
        ax = fig.add_subplot(gs[0, col])
        # Further left and higher than the article default: these panels carry a
        # two-line caption, and at the default offset "(a)" lands on top of it.
        add_panel_label(ax, next(panel_letters), x=-0.22, y=1.05)
        emb = PCA(n_components=2, random_state=SEED).fit_transform(X)
        lab = np.asarray(labels[name])
        for cl in np.unique(lab):
            m = lab == cl
            color, marker = cluster_style(int(cl))
            ax.scatter(emb[m, 0], emb[m, 1], c=color, marker=marker,
                       s=7, alpha=0.65, linewidths=0)
        observed, null = against_null[name]
        pval = (int(np.sum(null >= observed)) + 1) / (len(null) + 1)
        panel_caption(ax, f"$S$ = {observed:.2f}, p = {pval:.3f} (null {null.mean():.2f})", size=6.8)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel("PC 1", fontsize=6.5); ax.set_ylabel("PC 2", fontsize=6.5)

    matrix_axes = []
    for col, matrix in enumerate([rep_ari, alg_ari]):
        ax = fig.add_subplot(gs[1, col])
        matrix_axes.append(ax)
        add_panel_label(ax, next(panel_letters), x=-0.30, y=1.03)
        values = matrix.values
        im = ax.imshow(values, cmap=HEATMAP, vmin=-0.1, vmax=1.0)
        ax.set_xticks(range(len(matrix))); ax.set_yticks(range(len(matrix)))
        ax.set_xticklabels(matrix.columns, fontsize=6, rotation=30, ha="right")
        ax.set_yticklabels(matrix.index, fontsize=6)
        for i in range(len(matrix)):
            for j in range(len(matrix)):
                ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center",
                        fontsize=6.5,
                        color="white" if values[i, j] > 0.55 else SLATE)
        ax.grid(False)

    # The third cell of the bottom row states the lesson, and only the lesson.
    # A green check marks the question the data answers cleanly and a red cross
    # the one it does not; the caption spells out what each stands for. Helvetica
    # has no check or ballot-X glyph, so those two characters are set in DejaVu
    # Sans, which has them, while everything else keeps the manuscript font.
    ax = fig.add_subplot(gs[1, 2])
    ax.axis("off")

    def point(y, question, answer, mark):
        if mark == "check":
            ax.scatter(0.035, y, marker="s", s=230, color="#2e9c47",
                       transform=ax.transAxes, clip_on=False, zorder=3)
            ax.text(0.035, y, "\u2713", transform=ax.transAxes, color="white",
                    fontsize=8.5, fontweight="bold", ha="center", va="center",
                    zorder=4, fontfamily="DejaVu Sans")
        else:
            ax.text(0.035, y, "\u2717", transform=ax.transAxes, color="#d32f2f",
                    fontsize=15, fontweight="bold", ha="center", va="center",
                    zorder=4, fontfamily="DejaVu Sans")
        ax.text(0.14, y, question, transform=ax.transAxes, fontsize=7.0,
                va="center", ha="left", color=SLATE, weight="bold")
        ax.text(0.14, y - 0.10, answer, transform=ax.transAxes, fontsize=7.0,
                va="center", ha="left", color=SLATE)

    point(0.90, '"Are there clusters?"', "yes in (a) and (b), no in (c)", "check")
    point(0.57, '"What are the clusters?"', "no two choices agree   (d), (e)", "cross")

    # The scale belongs to (d) and (e), so it is attached to them and not to
    # this text panel, where it read as part of the summary.
    cb = fig.colorbar(im, ax=matrix_axes, fraction=0.055, pad=0.04,
                      label="adjusted Rand index")
    cb.outline.set_edgecolor(SLATE)
    cb.ax.tick_params(labelsize=6)

    outdir = savefig(fig, "clustering_illusion_molecules")
    print(f"wrote clustering_illusion_molecules.png/.pdf to {outdir}")
    print("   representation ARI:\n" + rep_ari.round(3).to_string())
    print("   algorithm ARI:\n" + alg_ari.round(3).to_string())


try:
    fig_silhouette_selection_bias()
    fig_silhouette_selection_bias(annotate=False)
    fig_aspirin_dihedral()
    fig_clustering_illusion_molecules()

    plt.show()
finally:
    # Leave the notebook as we found it, so anything you add below gets the
    # screen-sized type back.
    set_style()

### Exercises

1. Repeat the trajectory clustering of Section 4 with `stride=1` (all 10,000 frames) instead
   of `stride=10`. Bootstrap stability will rise, but the autocorrelation cell showed the
   dihedral has a statistical inefficiency near 300, so those extra frames are near-duplicates
   in the coordinate that matters. Estimate the effective number of independent frames (roughly
   `n / g`) at each stride, and say which of the two, the higher stability or the effective
   sample size, reflects the chemistry and which reflects the sampling.

2. Add a fifth representation to Section 7: RDKit physicochemical descriptors, standardized,
   clustered with k-means. Does it agree with any of the fingerprint partitions? Then answer in
   one sentence: if you had to report a single clustering of this library, which would you
   choose and how would you justify it in a paper?

In [ ]:
# YOUR CODE HERE

_Your answer:_